# Phase 4 — Tier-A Novelty Contributions

Four research-grade additions on top of the Phase-3 cross-attention model:

1. **Conformal prediction** — distribution-free SOH intervals with finite-sample coverage guarantees (`src/phase4/conformal.py`).
2. **Physics-informed loss** — monotonicity, Arrhenius temperature coupling, and capacity-resistance coupling penalties (`src/phase4/pinn_loss.py`).
3. **Weibull survival head** — Remaining-Useful-Life (RUL) prediction reframed as a censored time-to-event problem (`src/phase4/survival_head.py`).
4. **Retrieval-Augmented Prognostics (RAP)** — replace rule-based feature-to-text with K-NN retrieval + LLM generation (`src/phase4/rap.py`).

All four are *additive*: they extend the existing model rather than replacing it, so prior notebooks and the API still run unchanged.

## 0. Setup

In [ ]:
import importlib.util, os, sys, warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# Portable project-root setup (same pattern as the other notebooks)
def _load_setup():
    cwd = Path.cwd().resolve()
    candidates = [cwd / 'notebooks' / 'notebook_setup.py', cwd / 'notebook_setup.py']
    if cwd.name == 'notebooks':
        candidates.insert(0, cwd / 'notebook_setup.py')
    for fp in candidates:
        if fp.is_file():
            spec = importlib.util.spec_from_file_location('_notebook_setup', fp)
            mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
            return mod
    raise FileNotFoundError('notebooks/notebook_setup.py missing')

PROJECT_ROOT = _load_setup().setup()
print('PROJECT_ROOT =', PROJECT_ROOT)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
np.random.seed(42); torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device =', device)

In [ ]:
# Load the combined NASA + Mendeley feature corpus produced in Phase 3.
features_path = 'data/phase3/combined_nasa_mendeley.csv'
if not os.path.exists(features_path):
    features_path = 'data/phase2/harmonized/all_features.csv'
df = pd.read_csv(features_path).reset_index(drop=True)
print(f'Loaded {len(df)} cycles from {features_path}')
exclude_cols = {'battery_id','cycle_number','original_index','soh','has_spikes','dataset','test_condition','round_number'}
feature_cols = [c for c in df.columns if c not in exclude_cols]
print(f'Feature cols ({len(feature_cols)}):', feature_cols)

In [ ]:
# Per-battery time-ordered split: 60% train / 15% cal / 10% val / 15% test.
# The calibration split is what gives conformal prediction its guarantee.
splits = {'train': [], 'cal': [], 'val': [], 'test': []}
for bat_id, group in df.groupby('battery_id'):
    g = group.sort_values('cycle_number').reset_index(drop=True)
    n = len(g)
    i_train = int(n * 0.60); i_cal = int(n * 0.75); i_val = int(n * 0.85)
    splits['train'].append(g.iloc[:i_train])
    splits['cal'].append(g.iloc[i_train:i_cal])
    splits['val'].append(g.iloc[i_cal:i_val])
    splits['test'].append(g.iloc[i_val:])
for k in splits: splits[k] = pd.concat(splits[k], ignore_index=True)
print({k: len(v) for k, v in splits.items()})

In [ ]:
# Build numeric pseudo-sequences (matches the Phase-3 input convention exactly).
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(np.nan_to_num(splits['train'][feature_cols].values))
def to_seq(arr, seq_len=50):
    arr = np.nan_to_num(arr); arr = scaler.transform(arr)
    seq = np.tile(arr[:, None, :], (1, seq_len, 1))
    seq *= np.linspace(1.0, 0.95, seq_len)[None, :, None]
    return seq.astype(np.float32)
X_seq = {k: to_seq(splits[k][feature_cols].values) for k in splits}
y     = {k: splits[k]['soh'].values.astype(np.float32) for k in splits}
print({k: v.shape for k, v in X_seq.items()})

In [ ]:
# Cache SBERT embeddings on disk so re-runs are instant.
from feature_to_text import convert_features_to_text
EMB_CACHE = Path('data/phase4/embeddings_cache.npz')
EMB_CACHE.parent.mkdir(parents=True, exist_ok=True)

def _build_texts(frame):
    return convert_features_to_text(frame)['full_description'].tolist()

if EMB_CACHE.exists():
    blob = np.load(EMB_CACHE, allow_pickle=True)
    emb = {k: blob[k] for k in ('train','cal','val','test')}
    train_texts = list(blob['train_texts'])
    print('Loaded cached embeddings.')
else:
    from sentence_transformers import SentenceTransformer
    sbert = SentenceTransformer('all-mpnet-base-v2')
    texts = {k: _build_texts(splits[k]) for k in splits}
    emb = {k: sbert.encode(texts[k], show_progress_bar=(k=='train')).astype(np.float32) for k in splits}
    train_texts = texts['train']
    np.savez_compressed(EMB_CACHE, train_texts=np.array(train_texts, dtype=object), **emb)
    print('Cached embeddings to', EMB_CACHE)
{k: v.shape for k, v in emb.items()}

In [ ]:
# Load the saved Phase-3 cross-attention model so the demos start from a real, trained baseline.
from cross_attention_model import CrossAttentionSOHModel
ckpt_path = Path('models/cross_attention_model.pt')
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
cfg = ckpt['config']
base_model = CrossAttentionSOHModel(
    numeric_input_dim=cfg['numeric_input_dim'],
    semantic_input_dim=cfg['semantic_input_dim'],
    hidden_dim=cfg['hidden_dim'],
    n_attention_heads=cfg['n_attention_heads'],
    n_transformer_layers=cfg['n_transformer_layers'],
).to(device)
base_model.load_state_dict(ckpt['model_state_dict'])
base_model.eval()
print('Loaded base cross-attention model from', ckpt_path)

In [ ]:
@torch.no_grad()
def predict_soh(seq_np, emb_np, batch=128):
    base_model.eval()
    preds = []
    for i in range(0, len(seq_np), batch):
        s = torch.from_numpy(seq_np[i:i+batch]).to(device)
        e = torch.from_numpy(emb_np[i:i+batch]).to(device)
        preds.append(base_model(s, e)['soh'].cpu().numpy())
    return np.concatenate(preds)

preds = {k: predict_soh(X_seq[k], emb[k]) for k in splits}
for k in ('train','cal','val','test'):
    err = preds[k] - y[k]
    print(f'{k:5s}  MAE={np.mean(np.abs(err)):6.3f}  RMSE={np.sqrt(np.mean(err**2)):6.3f}')

## 1. Conformal prediction — calibrated SOH intervals

We ask for **90% coverage** (alpha = 0.10). Marginal split-conformal gives the standard guarantee; Mondrian split-conformal gives per-battery conditional coverage.

In [ ]:
from phase4.conformal import (SplitConformalRegressor, MondrianConformalRegressor,
                               empirical_coverage, grouped_empirical_coverage)

# The wrapper takes any callable that maps inputs -> point predictions.
# We feed it (seq, emb) bundled as a tuple so the calibration set is fully held-out.
def predict_pair(pair):
    seq_np, emb_np = pair
    return predict_soh(seq_np, emb_np)

marginal = SplitConformalRegressor(predict_pair, alpha=0.10)
marginal.calibrate((X_seq['cal'], emb['cal']), y['cal'])
interval_test = marginal.predict_interval((X_seq['test'], emb['test']))
print('Marginal split-conformal:')
print('  q_hat (interval half-width) =', round(marginal.q_hat, 3), 'SOH%')
print(' ', empirical_coverage(interval_test, y['test']))

In [ ]:
mondrian = MondrianConformalRegressor(predict_pair, alpha=0.10)
mondrian.calibrate((X_seq['cal'], emb['cal']), y['cal'], splits['cal']['battery_id'].values)
interval_test_m = mondrian.predict_interval((X_seq['test'], emb['test']), splits['test']['battery_id'].values)
print('Mondrian (per-battery) split-conformal:')
print(' ', empirical_coverage(interval_test_m, y['test']))
print('  Per-battery coverage:')
for bat, stats in grouped_empirical_coverage(interval_test_m, y['test'], splits['test']['battery_id'].values).items():
    print(f'    {bat:>10s}  cov={stats["empirical_coverage"]:.3f}  width={stats["mean_width"]:.2f}  n={stats["n"]}')

In [ ]:
# Visualise the calibrated interval against the truth.
order = np.argsort(y['test'])
fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(range(len(order)), interval_test.lower[order], interval_test.upper[order],
                alpha=0.25, label=f'90% conformal interval (\u00b1{marginal.q_hat:.2f})')
ax.plot(interval_test.point[order], '.', ms=3, label='Point prediction')
ax.plot(y['test'][order], 'r-', lw=1, label='True SOH')
ax.set_xlabel('Test sample (sorted by true SOH)'); ax.set_ylabel('SOH (%)')
ax.set_title('Conformal-calibrated SOH intervals on held-out test set')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
Path('results/phase4').mkdir(parents=True, exist_ok=True)
plt.savefig('results/phase4/conformal_intervals.png', dpi=150); plt.show()

## 2. Physics-informed loss — does it improve generalisation?

We train a fresh model with the composite PINN loss (MSE + monotonicity + Arrhenius + capacity-resistance) for a small number of epochs and compare against an MSE-only run. This is a quick demonstration; the published ablation should sweep weights.

In [ ]:
from phase4.pinn_loss import PINNCompositeLoss, PINNWeights
from torch.utils.data import Dataset, DataLoader

class PINNDataset(Dataset):
    def __init__(self, frame, X_seq, X_emb, y):
        self.frame = frame.reset_index(drop=True); self.X_seq = X_seq; self.X_emb = X_emb; self.y = y
        # Stable integer codes for battery_id so we can pass them through DataLoader.
        cats = pd.Categorical(self.frame['battery_id'])
        self.bat_codes = cats.codes.astype(np.int64); self.bat_categories = list(cats.categories)
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        row = self.frame.iloc[i]
        return {
            'sequence': torch.from_numpy(self.X_seq[i]),
            'embedding': torch.from_numpy(self.X_emb[i]),
            'soh': torch.tensor(self.y[i], dtype=torch.float32),
            'cycle': torch.tensor(row['cycle_number'], dtype=torch.float32),
            'temp_mean': torch.tensor(row.get('temp_mean', 0.0), dtype=torch.float32),
            'capacity':  torch.tensor(row.get('capacity', 0.0), dtype=torch.float32),
            'dcir':      torch.tensor(row.get('dcir', 0.0), dtype=torch.float32),
            'battery':   torch.tensor(self.bat_codes[i], dtype=torch.long),
        }

train_ds = PINNDataset(splits['train'], X_seq['train'], emb['train'], y['train'])
val_ds   = PINNDataset(splits['val'],   X_seq['val'],   emb['val'],   y['val'])
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False)
len(train_ds), len(val_ds)

In [ ]:
def fresh_model():
    return CrossAttentionSOHModel(
        numeric_input_dim=len(feature_cols), semantic_input_dim=emb['train'].shape[1],
        hidden_dim=cfg['hidden_dim'], n_attention_heads=cfg['n_attention_heads'],
        n_transformer_layers=cfg['n_transformer_layers'],
    ).to(device)

def train_quick(loss_kind, epochs=15):
    torch.manual_seed(7)
    model = fresh_model()
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    if loss_kind == 'mse':
        crit = torch.nn.MSELoss()
    else:
        crit = PINNCompositeLoss(weights=PINNWeights(monotonicity=0.05, arrhenius=0.02, capacity_resistance=0.05))
    history = {'train': [], 'val_mae': []}
    for ep in range(epochs):
        model.train(); losses = []
        for b in train_loader:
            opt.zero_grad()
            out = model(b['sequence'].to(device), b['embedding'].to(device))['soh']
            if loss_kind == 'mse':
                loss = crit(out, b['soh'].to(device))
            else:
                loss = crit(out, b['soh'].to(device), b['cycle'].to(device), b['temp_mean'].to(device),
                            b['capacity'].to(device), b['dcir'].to(device), b['battery'].to(device))
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            losses.append(loss.item())
        model.eval()
        with torch.no_grad():
            val_preds = np.concatenate([
                model(b['sequence'].to(device), b['embedding'].to(device))['soh'].cpu().numpy() for b in val_loader
            ])
        val_mae = float(np.mean(np.abs(val_preds - y['val'])))
        history['train'].append(float(np.mean(losses))); history['val_mae'].append(val_mae)
    return model, history

print('Training MSE-only baseline...')
model_mse, hist_mse = train_quick('mse')
print('Training PINN-augmented model...')
model_pinn, hist_pinn = train_quick('pinn')

In [ ]:
@torch.no_grad()
def quick_eval(model, seq_np, emb_np, y_true):
    model.eval(); preds = []
    for i in range(0, len(seq_np), 128):
        preds.append(model(torch.from_numpy(seq_np[i:i+128]).to(device), torch.from_numpy(emb_np[i:i+128]).to(device))['soh'].cpu().numpy())
    p = np.concatenate(preds)
    return {'MAE': float(np.mean(np.abs(p-y_true))), 'RMSE': float(np.sqrt(np.mean((p-y_true)**2))), 'preds': p}

mse_eval  = quick_eval(model_mse,  X_seq['test'], emb['test'], y['test'])
pinn_eval = quick_eval(model_pinn, X_seq['test'], emb['test'], y['test'])

# Monotonicity violation rate on the test set (per-battery in-order).
def mono_violations(preds, frame):
    bad = total = 0
    for _, g in frame.assign(_pred=preds).groupby('battery_id'):
        gg = g.sort_values('cycle_number')['_pred'].values
        if len(gg) < 2: continue
        bad += int(np.sum(gg[1:] - gg[:-1] > 0)); total += len(gg) - 1
    return bad / max(total, 1)

print('--- Test set ---')
print(f"MSE-only:  MAE={mse_eval['MAE']:.3f}  RMSE={mse_eval['RMSE']:.3f}  mono_viol={mono_violations(mse_eval['preds'], splits['test']):.3f}")
print(f"PINN:      MAE={pinn_eval['MAE']:.3f}  RMSE={pinn_eval['RMSE']:.3f}  mono_viol={mono_violations(pinn_eval['preds'], splits['test']):.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist_mse['val_mae'], label='MSE-only'); axes[0].plot(hist_pinn['val_mae'], label='PINN composite')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Validation MAE'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_title('Validation MAE during training')
ax = axes[1]
for name, ev in [('MSE-only', mse_eval), ('PINN', pinn_eval)]:
    ax.scatter(y['test'], ev['preds'], alpha=0.4, s=12, label=f"{name} (MAE={ev['MAE']:.2f})")
lo, hi = float(y['test'].min()), float(y['test'].max())
ax.plot([lo, hi], [lo, hi], 'k--', lw=1)
ax.set_xlabel('True SOH'); ax.set_ylabel('Predicted SOH'); ax.set_title('Test predictions'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('results/phase4/pinn_vs_mse.png', dpi=150); plt.show()

## 3. Weibull RUL head — survival-style remaining-useful-life

Add a two-output (scale, shape) Weibull head on the *frozen* base cross-attention model. We train only the head with a censoring-aware Weibull NLL on cycles whose battery does (or does not) reach 80% SOH within the recording.

In [ ]:
from phase4.survival_head import (CrossAttentionWithRULHead, weibull_nll, summarise_predictions,
                                   compute_rul_targets)

rul_df = compute_rul_targets(df, eol_threshold=80.0)
rul_lookup = rul_df.set_index(['battery_id', 'cycle_number'])[['rul', 'event']]
for k in splits:
    keys = list(zip(splits[k]['battery_id'], splits[k]['cycle_number']))
    splits[k] = splits[k].assign(
        rul   = rul_lookup.loc[keys, 'rul'].values,
        event = rul_lookup.loc[keys, 'event'].values,
    )
splits['train'][['battery_id','cycle_number','soh','rul','event']].head()

In [ ]:
rul_model = CrossAttentionWithRULHead(base_model).to(device)
rul_model.freeze_base()  # train only the survival head
opt = torch.optim.Adam(rul_model.rul_head.parameters(), lr=5e-3)

def make_loader(k, batch=64, shuffle=True):
    s = splits[k]
    items = list(zip(X_seq[k], emb[k], s['rul'].values.astype(np.float32), s['event'].values.astype(np.int64)))
    def collate(batch_items):
        seq = torch.tensor(np.stack([b[0] for b in batch_items]))
        em  = torch.tensor(np.stack([b[1] for b in batch_items]))
        ru  = torch.tensor(np.array([b[2] for b in batch_items]))
        ev  = torch.tensor(np.array([b[3] for b in batch_items]))
        return seq, em, ru, ev
    return torch.utils.data.DataLoader(items, batch_size=batch, shuffle=shuffle, collate_fn=collate)

rul_train = make_loader('train'); rul_val = make_loader('val', shuffle=False)
for ep in range(20):
    rul_model.train(); losses = []
    for seq, em, ru, ev in rul_train:
        seq, em, ru, ev = seq.to(device), em.to(device), ru.to(device), ev.to(device)
        opt.zero_grad()
        out = rul_model(seq, em)
        loss = weibull_nll(out['rul_scale'], out['rul_shape'], ru, ev)
        if not torch.isfinite(loss): continue
        loss.backward(); torch.nn.utils.clip_grad_norm_(rul_model.rul_head.parameters(), 1.0); opt.step()
        losses.append(loss.item())
    if (ep+1) % 5 == 0:
        print(f'epoch {ep+1:2d}  train NLL={np.mean(losses):.3f}')

In [ ]:
rul_model.eval()
all_scale, all_shape, all_rul, all_event = [], [], [], []
with torch.no_grad():
    for seq, em, ru, ev in make_loader('test', shuffle=False):
        out = rul_model(seq.to(device), em.to(device))
        all_scale.append(out['rul_scale'].cpu()); all_shape.append(out['rul_shape'].cpu())
        all_rul.append(ru); all_event.append(ev)
scale = torch.cat(all_scale); shape = torch.cat(all_shape)
rul_true = torch.cat(all_rul).numpy(); event = torch.cat(all_event).numpy()
summary = summarise_predictions(scale, shape)

# MAE on uncensored ground-truth only (censored RULs are lower bounds, not point targets).
obs = event == 1
if obs.any():
    mae_med = float(np.mean(np.abs(summary.median[obs] - rul_true[obs])))
    coverage_80 = float(np.mean((rul_true[obs] >= summary.p10[obs]) & (rul_true[obs] <= summary.p90[obs])))
    print(f'Uncensored test cycles: n={obs.sum()}  median-RUL MAE={mae_med:.2f}  empirical 80% coverage={coverage_80:.3f}')
else:
    print('No uncensored test cycles \u2014 evaluating on censored cycles only.')

# Distribution of predicted shape parameter (k > 1 = wear-out, k < 1 = infant mortality).
print(f'predicted Weibull shape: mean={float(shape.mean()):.2f}  median={float(shape.median()):.2f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
obs = event == 1
if obs.any():
    order = np.argsort(rul_true[obs])
    ax.fill_between(range(int(obs.sum())), summary.p10[obs][order], summary.p90[obs][order], alpha=0.25, label='80% predictive interval')
    ax.plot(summary.median[obs][order], '.', ms=3, label='Predicted median RUL')
    ax.plot(rul_true[obs][order], 'r-', lw=1, label='True RUL')
    ax.set_xlabel('Test sample (sorted by true RUL)'); ax.set_ylabel('Cycles to EoL')
    ax.set_title('Weibull RUL predictions on uncensored test cycles')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig('results/phase4/rul_predictions.png', dpi=150); plt.show()
else:
    print('Nothing to plot \u2014 the test split contained no uncensored cycles.')

## 4. Retrieval-Augmented Prognostics (RAP)

Build a NumPy/FAISS K-NN index over the training corpus and demonstrate the retrieve-then-generate pipeline. The LLM call is cached on disk so subsequent runs are deterministic. If `OPENAI_API_KEY` is unset, RAP falls back to the rule-based template generator and still produces an answer.

In [ ]:
from phase4.rap import build_rap, LLMConfig
from feature_to_text import FeatureToText

# Reuse the cached training texts from earlier.
if not train_texts:
    train_texts = _build_texts(splits['train'])

fallback_converter = FeatureToText()
fallback_converter.set_reference_values(splits['train'].iloc[0].to_dict())
def template_fallback(feats):
    return fallback_converter.convert_single(feats).full_description

rap = build_rap(
    train_features_df=splits['train'].reset_index(drop=True),
    feature_cols=feature_cols,
    descriptions=train_texts,
    k=5,
    cache_dir='data/phase4/rap_cache',
    llm_config=LLMConfig(model=os.environ.get('OPENAI_MODEL', 'gpt-4o-mini')),
    template_fallback=template_fallback,
)
print('RAP ready. OpenAI key present:', bool(os.environ.get('OPENAI_API_KEY')))

In [ ]:
# Show RAP on three example test cycles spanning the SOH range.
for label, idx in [('healthy', int(np.argmax(y['test']))),
                   ('mid-life', int(np.argsort(y['test'])[len(y['test'])//2])),
                   ('degraded', int(np.argmin(y['test'])))]:
    row = splits['test'].iloc[idx]
    diag = rap.diagnose(row.to_dict())
    print(f"\n=== {label}  battery={row['battery_id']}  cycle={int(row['cycle_number'])}  SOH={row['soh']:.1f}% ===")
    print('Used LLM:', diag.used_llm)
    print('Top retrieved analogues:')
    for h in diag.hits[:3]:
        print(f'  - {h.battery_id} cycle {h.cycle_number}: SOH={h.soh:.1f}%  cosine_dist={h.distance:.3f}')
    print('Generated diagnosis:')
    print(' ', diag.text)

In [ ]:
# Quantitative evaluation: do RAP-generated descriptions, embedded with SBERT and used as the
# semantic input, beat the template descriptions for the downstream regressor on the test set?
# We only embed a sample of the test set to keep this fast and avoid burning the LLM budget.
from sentence_transformers import SentenceTransformer
sbert = SentenceTransformer('all-mpnet-base-v2')

RAP_SAMPLE = min(60, len(splits['test']))
sample_idx = np.random.RandomState(0).choice(len(splits['test']), size=RAP_SAMPLE, replace=False)
sample_rows = splits['test'].iloc[sample_idx].reset_index(drop=True)
rap_texts = [rap.diagnose(r.to_dict()).text for _, r in sample_rows.iterrows()]
tpl_texts = _build_texts(sample_rows)

rap_emb = sbert.encode(rap_texts, show_progress_bar=False).astype(np.float32)
tpl_emb = sbert.encode(tpl_texts, show_progress_bar=False).astype(np.float32)
sample_seq = X_seq['test'][sample_idx]; sample_y = y['test'][sample_idx]

rap_metrics = quick_eval(base_model, sample_seq, rap_emb, sample_y)
tpl_metrics = quick_eval(base_model, sample_seq, tpl_emb, sample_y)
print(f"Template descriptions  ->  MAE={tpl_metrics['MAE']:.3f}  RMSE={tpl_metrics['RMSE']:.3f}")
print(f"RAP descriptions       ->  MAE={rap_metrics['MAE']:.3f}  RMSE={rap_metrics['RMSE']:.3f}")

## Summary

| Contribution | Where | What's novel |
|---|---|---|
| Conformal prediction | `src/phase4/conformal.py` | Marginal + Mondrian split-conformal SOH intervals with finite-sample coverage; reports per-battery conditional coverage. |
| Physics-informed loss | `src/phase4/pinn_loss.py` | Soft-monotonicity, Arrhenius rank consistency, capacity-resistance coupling penalties combined with MSE. |
| Weibull survival head | `src/phase4/survival_head.py` | Two-parameter Weibull head with right-censoring-aware NLL for full RUL distributions. |
| Retrieval-Augmented Prognostics | `src/phase4/rap.py` | K-NN-conditioned LLM diagnosis generator that drops into the existing SBERT pipeline; cached + has template fallback. |

Saved artefacts: `results/phase4/conformal_intervals.png`, `results/phase4/pinn_vs_mse.png`, `results/phase4/rul_predictions.png`.